# Numerical Differentiation

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 06.02 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/03_numerical_differentiation.ipynb)

---

## 🎯 Learning Objective

Compare finite-difference approximations to automatic differentiation (forward and reverse mode), and use finite differences to gradient-check an implementation.

---

## 📐 Theory

`02.04` derived backpropagation as reverse-mode automatic differentiation. This notebook fills
in the two things that discussion deferred: what finite differences are (the crude, first
tool anyone reaches for) and what *forward-mode* AD is (the other exact, non-symbolic option) —
and why reverse mode specifically is what deep learning needs.

### Finite differences

The most direct way to estimate $f'(x)$ numerically is to go back to the definition and use a
small but finite $h$ instead of a true limit:

$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad \text{(forward difference)}, \qquad
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad \text{(central difference)}$$

Central differences are more accurate (error shrinks like $O(h^2)$ instead of $O(h)$, since the
$O(h)$ error terms cancel by symmetry) but both share the same fundamental tension: shrinking
$h$ reduces *truncation* error (how well the finite difference approximates the true limit) but
increases *rounding* error (subtracting two very close floating-point numbers, recall `06.02`'s
catastrophic cancellation), so there's an optimal $h$ that's neither too big nor too small —
finite differences are an *approximation*, never exact.

### Automatic differentiation: exact, not approximate

**Automatic differentiation (AD)** computes derivatives that are exact up to floating-point
rounding — no truncation error at all — by mechanically applying the chain rule to the sequence
of elementary operations a program actually performs, rather than approximating a limit. AD
comes in two flavors, differing in which direction information flows through the computation:

**Forward mode** propagates a derivative *alongside* each intermediate value, in the same order
the original computation runs. A clean way to implement this: represent every number as a
**dual number** $a + b\varepsilon$, where $\varepsilon$ is a symbol satisfying $\varepsilon^2=0$
(not zero itself, but its square vanishes). Arithmetic on dual numbers automatically tracks
derivatives: $(a+b\varepsilon)(c+d\varepsilon) = ac + (ad+bc)\varepsilon$ (the $bd\varepsilon^2$
term vanishes), which is exactly the product rule ($\frac{d}{dx}(uv) = u'v + uv'$) if $b, d$ are
$u', v'$. Seed $x$ as $x_0 + 1\cdot\varepsilon$ and every subsequent dual number's $\varepsilon$
coefficient is automatically $df/dx$ at that point — no separate backward pass needed.

**Reverse mode** (recall `02.04`'s full derivation) instead runs a complete forward pass first,
then walks the recorded computation graph backward, propagating $\partial(\text{output})/\partial(\text{node})$
using the local-gradient recipe.

### Forward vs. reverse: why direction matters at scale

For $f:\mathbb{R}^n\to\mathbb{R}^m$: **forward mode** costs one pass per *input* dimension and
gets the derivative with respect to every output at once — efficient when $n$ is small, $m$ is
large. **Reverse mode** costs one pass per *output* dimension and gets the derivative with
respect to every input at once — efficient when $m$ is small, $n$ is large. A neural network's
loss function has $n$ = millions/billions of parameters and $m=1$ (a single scalar loss) — the
single most lopsided case imaginable, which is exactly why every deep learning framework uses
reverse mode (backprop) rather than forward mode: computing a billion forward-mode passes to
get one gradient would be absurd, while one reverse-mode pass gets the whole gradient at once.

### Gradient checking

Because finite differences and AD are computed by entirely different mechanisms, comparing
them is a standard way to catch bugs in a hand-written backward pass: if your analytic gradient
implementation disagrees with a central-difference estimate by more than the expected numerical
tolerance, you have a bug, not a rounding artifact. This is exactly the "verify by an
independent method" discipline used throughout this repo's own from-scratch implementations.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting finite-difference error against step size $h$ on a log-log scale reveals the classic
U-shape: too-large $h$ has truncation error, too-small $h$ has rounding error, and there's a
sweet spot in between.

In [ ]:
# f(x) = sin(x), f'(x) = cos(x). Sweep h and measure finite-difference error against the exact
# analytic derivative -- this reveals the truncation-vs-rounding tradeoff directly.
f = np.sin
f_prime_exact = np.cos
x0 = 1.0
true_deriv = f_prime_exact(x0)

hs = np.logspace(-16, 0, 200)
forward_errors = np.abs((f(x0 + hs) - f(x0)) / hs - true_deriv)
central_errors = np.abs((f(x0 + hs) - f(x0 - hs)) / (2 * hs) - true_deriv)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.loglog(hs, forward_errors, color="#C44E52", lw=2, label="forward difference (O(h) truncation)")
ax.loglog(hs, central_errors, color="#4C72B0", lw=2, label="central difference (O(h^2) truncation)")
ax.set_xlabel("step size h"); ax.set_ylabel("|error| vs. true derivative")
ax.set_title("Finite differences: too-large h truncates, too-small h loses precision\n(rounding error, recall 06.02)")
ax.legend()
plt.show()

best_h_central = hs[np.argmin(central_errors)]
best_h_forward = hs[np.argmin(forward_errors)]
print(f"Best step size for central differences: h ~ {best_h_central:.2e}, error = {central_errors.min():.2e}")
print(f"Best step size for forward differences: h ~ {best_h_forward:.2e}, error = {forward_errors.min():.2e}")
print(f"Central differences reach a smaller minimum error using a LARGER optimal h --")
print(f"its faster O(h^2) truncation convergence lets it stop shrinking h sooner, before rounding error dominates.")

## 🐍 Implementation from Scratch

We build a minimal forward-mode AD engine using dual numbers, verify it against both finite
differences and `torch.autograd`'s reverse mode on the same function, then use finite
differences to gradient-check a hand-written backward pass.

In [ ]:
class Dual:
    """a + b*epsilon, with epsilon^2 = 0 by definition. Arithmetic on these numbers
    automatically carries a derivative alongside a value -- no separate backward pass."""
    def __init__(self, real, dual=0.0):
        self.real, self.dual = real, dual

    def __add__(self, other):
        other = other if isinstance(other, Dual) else Dual(other, 0.0)
        return Dual(self.real + other.real, self.dual + other.dual)
    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Dual) else Dual(other, 0.0)
        # (a+be)(c+de) = ac + (ad+bc)e + bd*e^2 = ac + (ad+bc)e, since e^2=0 -- this IS the
        # product rule, derived purely from epsilon^2=0, not assumed separately
        return Dual(self.real * other.real, self.real * other.dual + self.dual * other.real)
    __rmul__ = __mul__

    def sin(self):
        return Dual(np.sin(self.real), np.cos(self.real) * self.dual)

# f(x) = x^2 + 3x, an easy case to check by hand: f'(x) = 2x + 3
def f_dual(x):
    return x * x + Dual(3.0) * x

x0 = Dual(2.0, 1.0)  # seed dual=1.0 to ask for df/dx directly
result = f_dual(x0)
print(f"Forward-mode AD:  f(2)={result.real}, f'(2)={result.dual}")

h = 1e-6
fd = (lambda x: x**2 + 3*x)
finite_diff = (fd(2.0 + h) - fd(2.0 - h)) / (2 * h)
print(f"Central finite diff: f'(2)~{finite_diff:.6f}")

xt = torch.tensor(2.0, requires_grad=True)
(xt * xt + 3 * xt).backward()
print(f"torch.autograd (reverse mode): f'(2)={xt.grad.item()}")
print(f"All three agree: {np.allclose([result.dual, finite_diff, xt.grad.item()], 7.0, atol=1e-4)}")

# A harder case with a nonlinearity, to confirm the chain rule composes correctly through .sin()
def g_dual(x):
    return (x * x).sin()

x1 = Dual(1.5, 1.0)
res = g_dual(x1)
analytic = 2 * 1.5 * np.cos(1.5**2)  # d/dx[sin(x^2)] = cos(x^2) * 2x, by hand
print(f"\nsin(x^2) at x=1.5:  forward-mode AD f'={res.dual:.6f}, hand-derived={analytic:.6f}")
print(f"Match: {np.isclose(res.dual, analytic)}")

# --- Gradient checking: catch a DELIBERATELY WRONG hand-written gradient ---
def my_function(x):
    return np.sum(x**3)

def my_gradient_CORRECT(x):
    return 3 * x**2

def my_gradient_BUGGY(x):
    return 2 * x**2   # deliberately wrong coefficient, simulating a real typo

def gradient_check(f, grad_fn, x, h=1e-5):
    numeric_grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus, x_minus = x.copy(), x.copy()
        x_plus[i] += h; x_minus[i] -= h
        numeric_grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    analytic_grad = grad_fn(x)
    max_diff = np.max(np.abs(numeric_grad - analytic_grad))
    return max_diff, numeric_grad, analytic_grad

x_test = np.array([1.0, 2.0, 3.0])
diff_correct, _, _ = gradient_check(my_function, my_gradient_CORRECT, x_test)
diff_buggy, _, _ = gradient_check(my_function, my_gradient_BUGGY, x_test)
print(f"\nGradient check, CORRECT implementation: max diff = {diff_correct:.2e}  (passes)")
print(f"Gradient check, BUGGY implementation:    max diff = {diff_buggy:.2e}  (FAILS -- caught the bug)")

## 🤖 Why This Matters for AI

`torch.autograd` is exactly the reverse-mode AD engine described above, scaled from scalars to
tensors and vectorized. When implementing a custom `torch.autograd.Function` with a hand-written
backward pass (common when writing a new, faster kernel for an existing operation), the standard
sanity check is exactly the gradient-checking technique above: PyTorch even ships it built in as
`torch.autograd.gradcheck`, which compares your analytic backward pass against finite differences
and raises an error if they disagree beyond a tolerance. Below, we use `torch.autograd.gradcheck`
directly on a custom autograd Function, first with a correct backward pass (passes) and then with
a deliberately introduced bug (fails) — confirming this isn't just a hand-rolled convenience but
a real, load-bearing tool used when extending PyTorch itself.

In [ ]:
class CubeCorrect(torch.autograd.Function):
    """A custom autograd Function for x^3, with a CORRECT hand-written backward pass."""
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 3

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * 3 * x ** 2   # d/dx(x^3) = 3x^2

class CubeBuggy(torch.autograd.Function):
    """The SAME operation, but with a deliberately wrong coefficient in the backward pass --
    simulating exactly the kind of typo gradient checking exists to catch."""
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 3

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * 2 * x ** 2   # BUG: should be 3*x**2, not 2*x**2

x_check = torch.randn(5, dtype=torch.float64, requires_grad=True)

correct_passes = torch.autograd.gradcheck(CubeCorrect.apply, (x_check,))
print(f"Correct backward pass: gradcheck passes = {correct_passes}")

try:
    torch.autograd.gradcheck(CubeBuggy.apply, (x_check,))
    print("Buggy backward pass: gradcheck passed (this should NOT happen)")
except RuntimeError:
    print("Buggy backward pass: gradcheck correctly RAISED an error -- the bug was caught")

print("\ntorch.autograd.gradcheck runs the exact same finite-difference comparison we built by")
print("hand above -- it's the production version of the same idea, used whenever PyTorch's own")
print("built-in operations need their hand-written backward passes verified.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, compute $f'(2)$ for $f(x) = x^3 - 2x$ using both a forward difference and a central
   difference, both with $h=0.1$. Compare both against the exact analytic value $f'(2)=10$ and
   check which is closer — it should be the one the Theory section's $O(h^2)$ vs. $O(h)$ claim
   predicts.

<details>
<summary>💡 Solution</summary>

$f(2)=4$, $f(2.1)=7.061$, $f(1.9)=3.059$. Forward: $(7.061-4)/0.1 = 10.61$, error $0.61$.
Central: $(7.061-3.059)/(2\times 0.1) = 10.01$, error $0.01$ — about $60\times$ smaller, matching
the theory that central differences' error shrinks like $O(h^2)$ while forward differences only
shrink like $O(h)$ for the same $h$.

</details>

### 💭 UNDERSTAND
2. The Visual Intuition plot shows finite-difference error against $h$ as a U-shape, with an
   optimal $h$ around $10^{-6}$ to $10^{-9}$ depending on the method — not $h\to 0$. Someone
   argues "just use the smallest $h$ your floating-point type allows, to minimize truncation
   error as much as possible." Why is this wrong, and what actually happens if you evaluate a
   central difference at, say, $h=10^{-20}$ instead of the optimal $h\approx 4\times10^{-6}$?

<details>
<summary>💡 Solution</summary>

Truncation error (how well the finite-difference formula approximates the true derivative)
shrinks as $h\to 0$, but *rounding* error grows as $h\to 0$: $f(x+h)$ and $f(x-h)$ become
numerically indistinguishable once $h$ is smaller than what the float format can resolve near
$x$, so the subtraction suffers catastrophic cancellation (recall `06.02`) and the division by
$2h$ amplifies whatever rounding noise is left. Concretely, evaluating the central difference of
$\sin(x)$ at $x=1$ with $h=10^{-20}$ returns exactly $0.0$ — $\sin(1+10^{-20})$ and
$\sin(1-10^{-20})$ both round to the *identical* float64 value, so the numerator is exactly zero
before the division even happens, giving a derivative estimate of $0$ instead of $\cos(1)\approx
0.5403$. Shrinking $h$ past the sweet spot doesn't just add noise — it can silently return a
completely wrong, confidently-reported-as-exact zero.

</details>

### ✏️ DERIVE
3. Derive the leading-order error term of the central-difference formula using a Taylor
   expansion of $f(x+h)$ and $f(x-h)$ around $x$, and confirm it matches the notebook's claim
   that central-difference error is $O(h^2)$.

<details>
<summary>💡 Solution outline</summary>

Expand both to fourth order: $f(x+h) = f(x) + hf'(x) + \frac{h^2}{2}f''(x) + \frac{h^3}{6}f'''(x) + O(h^4)$
and $f(x-h) = f(x) - hf'(x) + \frac{h^2}{2}f''(x) - \frac{h^3}{6}f'''(x) + O(h^4)$. Subtracting,
every even-order term cancels (that's *why* central differences are more accurate than forward
differences — the symmetry kills the $O(h)$ and $O(h^2)$ error terms that a one-sided difference
keeps): $f(x+h)-f(x-h) = 2hf'(x) + \frac{h^3}{3}f'''(x) + O(h^5)$. Dividing by $2h$:
$\frac{f(x+h)-f(x-h)}{2h} = f'(x) + \frac{h^2}{6}f'''(x) + O(h^4)$ — so the leading error term is
$\frac{h^2}{6}f'''(x)$, confirmed $O(h^2)$. Numerically, for $f=\sin$ at $x=1$
($f'''(x)=-\cos(1)$), the predicted error $\frac{h^2}{6}(-\cos 1)$ matches the actual central-difference
error to 4 significant figures at $h=0.01$ and $h=0.001$.

</details>

### 🐍 IMPLEMENT
4. Implement **complex-step differentiation**: for real-analytic $f$,
   $f'(x) \approx \text{Im}(f(x+ih))/h$ for a tiny real $h$, evaluated using complex arithmetic.
   Unlike finite differences, this formula has *no subtraction of near-equal numbers* at all (no
   catastrophic cancellation), so it remains accurate even as $h\to 0$. Test it on
   $f(x) = \text{silu}(x) = x\cdot\sigma(x)$ at $x=1$ for $h \in \{10^{-8}, 10^{-20}, 10^{-150}\}$.

<details>
<summary>✅ How to know you're right</summary>

Compare against the exact analytic derivative of silu,
$f'(x) = \sigma(x) + x\sigma(x)(1-\sigma(x))$. The complex-step result should stay accurate to
machine precision ($\text{error} \sim 10^{-16}$) at *every* one of the three wildly different
$h$ values, including $h=10^{-150}$ — in sharp contrast to a central difference at the same
tiny $h$ values, which returns exactly $0.0$ once $h$ is small enough that $x+h$ and $x-h$ round
to the same float. This is the actual point of the exercise: complex-step differentiation
doesn't have the truncation/rounding tradeoff at all, because there's no subtraction to cancel.

</details>

### 🤖 APPLY
5. You're debugging a custom `torch.autograd.Function` for a new attention-score kernel and its
   loss stays flat during training — a classic symptom of a backward-pass bug that returns
   (near-)zero gradients while the forward pass looks completely correct. Using
   `torch.autograd.gradcheck` (from the Why This Matters for AI section) as your primary
   diagnostic, describe the debugging workflow: what you'd check first, what a `gradcheck`
   failure tells you that a forward-pass unit test wouldn't, and what you'd do differently if
   `gradcheck` *passes* but the loss is still flat.

<details>
<summary>✅ What you should observe</summary>

`gradcheck` isolates the bug to the backward pass specifically: since it only ever compares your
`backward()`'s output against finite differences of the *forward* pass, a forward-pass unit test
checking output values would pass even if `backward()` is completely wrong (as the notebook's
`CubeBuggy` example shows — same forward pass, broken backward, and only `gradcheck` catches it).
If `gradcheck` fails, the fix is entirely inside your hand-written `backward()`. If `gradcheck`
*passes* but training still stalls, the bug is elsewhere in the pipeline the finite-difference
check can't see: e.g. a vanishing-gradient interaction with other layers, an optimizer or
learning-rate issue, or (recall `06.02`) a numerically unstable forward computation that IS
differentiated correctly but is differentiating a nearly-flat or saturated function.

</details>

### 🚀 CHALLENGE
6. `torch.autograd.gradcheck` above checks a *scalar-to-scalar* (or reduced-to-scalar) function.
   Design a `gradient_check` for a genuinely vector-valued function
   $\mathbf{f}(x,y) = (x^2y,\ xy^2)$, which has a full $2\times2$ Jacobian rather than a single
   gradient vector — checking each output coordinate's finite-difference derivative against each
   input coordinate separately. Then, generalize: for $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^m$,
   how many finite-difference evaluations does a full Jacobian check require, and how does that
   compare to the number of forward-mode vs. reverse-mode AD passes needed for the same full
   Jacobian? For $n=1000, m=1$ (a typical loss function) and for $n=1, m=1000$ (a typical
   per-token output layer), which AD mode wins in each case, and does gradient-*checking*
   (which needs the *full* Jacobian, not just one mode's efficient slice) still get to enjoy that
   asymmetry?

<details>
<summary>🔍 What a strong answer covers</summary>

A full 2D gradient check for $\mathbf{f}(x,y)=(x^2y, xy^2)$ perturbs $x$ and $y$ independently
and compares against all 4 entries of the true Jacobian
$J = \begin{pmatrix}2xy & x^2 \\ y^2 & 2xy\end{pmatrix}$; a strong answer verifies at least one
concrete point numerically. For general $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^m$, a full
finite-difference Jacobian needs $O(n)$ evaluations regardless of $m$ (perturb each input once,
read off all $m$ outputs from that one perturbed forward pass) — so finite-difference *checking*
cost is governed by $n$ alone, not by whichever of $n, m$ is smaller. Forward-mode AD needs $n$
passes (one per input direction) to assemble the full Jacobian; reverse-mode needs $m$ passes
(one per output direction). For $n=1000,m=1$: reverse mode wins decisively (1 pass vs. 1000) —
exactly the deep-learning-loss case. For $n=1,m=1000$: forward mode wins (1 pass vs. 1000) — a
per-token output layer with a scalar input direction. The interesting twist: gradient
*checking* itself doesn't get to exploit this asymmetry the way AD does, since finite differences
always cost $O(n)$ evaluations to get the full Jacobian needed for a *complete* check — so
checking a reverse-mode-favored function (large $n$) is inherently expensive by finite
differences even though computing its gradient via AD is cheap. In practice this is why
`gradcheck`-style tools are used on small test cases (a handful of input/output dims), not on
full-scale models.

</details>

---

## 📚 Further Reading
- Baydin et al., ["Automatic Differentiation in Machine Learning: a Survey"](https://arxiv.org/abs/1502.05767)
- PyTorch documentation, [`torch.autograd.gradcheck`](https://pytorch.org/docs/stable/generated/torch.autograd.gradcheck.html)

---

*Next notebook: [Numerical Linear Algebra](04_numerical_linear_algebra.ipynb)*